Скрипт создает .txt файлы для загрузки в деревья коллизий. Поддерживается проверка на столкновения между прямоугольниками и кругами.

В каждой строке хранится `x, y, vx, xy` - условие столкновения.

In [5]:
import math

MAX_POS = 25
MAX_VEL = 5

CIRCLE_RADIUS = 1

RECT_HALF_WIDTH = 1
RECT_HALF_HEIGHT = 2


def clamp(v, a, b):
    return max(a, min(v, b))


def point_segment_distance(px, py, ax, ay, bx, by):
    abx = bx - ax
    aby = by - ay

    apx = px - ax
    apy = py - ay

    ab_len_sq = abx * abx + aby * aby

    if ab_len_sq == 0:
        return math.sqrt(apx * apx + apy * apy)

    t = (apx * abx + apy * aby) / ab_len_sq
    t = clamp(t, 0.0, 1.0)

    cx = ax + abx * t
    cy = ay + aby * t

    dx = px - cx
    dy = py - cy

    return math.sqrt(dx * dx + dy * dy)


def collide_circle_circle(x, y, vx, vy):
    total_radius = CIRCLE_RADIUS * 2

    dist = point_segment_distance(
        0,
        0,
        x,
        y,
        x + vx,
        y + vy,
    )

    return dist <= total_radius


def distance_point_to_aabb(px, py, half_w, half_h):
    cx = clamp(px, -half_w, half_w)
    cy = clamp(py, -half_h, half_h)

    dx = px - cx
    dy = py - cy

    return math.sqrt(dx * dx + dy * dy)


def collide_circle_rectangle(x, y, vx, vy):
    STEPS = 20

    for i in range(STEPS + 1):

        t = i / STEPS

        px = x + vx * t
        py = y + vy * t

        dist = distance_point_to_aabb(
            px,
            py,
            RECT_HALF_WIDTH,
            RECT_HALF_HEIGHT,
        )

        if dist <= CIRCLE_RADIUS:
            return True

    return False


def collide_rectangle_rectangle(x, y, vx, vy):
    expanded_half_x = RECT_HALF_WIDTH * 2
    expanded_half_y = RECT_HALF_HEIGHT * 2

    x_inv_entry = (-expanded_half_x - x)
    x_inv_exit = (expanded_half_x - x)

    y_inv_entry = (-expanded_half_y - y)
    y_inv_exit = (expanded_half_y - y)

    if vx == 0:
        x_entry = -math.inf
        x_exit = math.inf
    else:
        x_entry = x_inv_entry / vx
        x_exit = x_inv_exit / vx

    if vy == 0:
        y_entry = -math.inf
        y_exit = math.inf
    else:
        y_entry = y_inv_entry / vy
        y_exit = y_inv_exit / vy

    entry_time = max(
        min(x_entry, x_exit),
        min(y_entry, y_exit),
    )

    exit_time = min(
        max(x_entry, x_exit),
        max(y_entry, y_exit),
    )

    return (
        entry_time <= exit_time
        and 0.0 <= entry_time <= 1.0
    )

def generate(filename, collision_fn):

    collisions = 0
    total = 0

    with open(filename, "w") as f:

        for x in range(-MAX_POS, MAX_POS + 1):

            for y in range(-MAX_POS, MAX_POS + 1):

                for vx in range(-MAX_VEL, MAX_VEL + 1):

                    for vy in range(-MAX_VEL, MAX_VEL + 1):

                        if vx == 0 and vy == 0:
                            continue

                        total += 1

                        if collision_fn(
                            x,
                            y,
                            vx,
                            vy,
                        ):

                            collisions += 1

                            f.write(
                                f"{x} {y} {vx} {vy}\n"
                            )

    print(f"{filename}")
    print(f"total checked: {total}")
    print(f"collisions: {collisions}")
    print()


generate(
    "collision_circle_circle.txt",
    collide_circle_circle,
)

generate(
    "collision_circle_rectangle.txt",
    collide_circle_rectangle,
)

generate(
    "collision_rectangle_rectangle.txt",
    collide_rectangle_rectangle,
)

collision_circle_circle.txt
total checked: 312120
collisions: 3624

collision_circle_rectangle.txt
total checked: 312120
collisions: 6820

collision_rectangle_rectangle.txt
total checked: 312120
collisions: 9128

